# 🏗️ CEM4644 · MP4 — Segmentation for progress and quantity measurement
## Workshop (in class): *Structural work on site: concrete, rebar, formwork, steel, scaffolding*

**No coding needed.** Each grey box below is one *step*: click the ▶ (play) button at its left, wait until it finishes, look at the result, then answer the report question that follows. Run the steps **from top to bottom**.

**What you will do (about 80 minutes)**
1. Look at site photos and the materials in them.
2. Ask a segmentation model, by name, for a material: see the mask, the overlay, and the share of the photo it covers.
3. Compare photos and follow a site through time.
4. Examine where the model goes wrong: wording, weak regions, and how to correct it.
5. Try your own photo and your own words.

**Before you start:** menu *Runtime → Change runtime type → T4 GPU → Save*. The model used here (SAM 3) is large: with a GPU each request takes well under a second; without one, precomputed results still work but live requests take about a minute each.

In [ ]:
#@title ▶ Step 0 · Run me first (2–3 minutes) { display-mode: "form" }
#@markdown Click ▶ and wait for the green ✅ line. This downloads the photos with their precomputed results and loads SAM 3 (about 3 GB).
#@markdown Untick *load_model* only if you have no GPU and want to skip the live steps.
load_model = True #@param {type:"boolean"}
import os, sys, subprocess
if not os.path.isdir("CEM4644/mp4_segmentation"):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "https://github.com/Haolan-Zhang/CEM4644.git"], check=True)
sys.path.insert(0, os.path.abspath("CEM4644/mp4_segmentation"))
from aec_seg import lab
lab.setup(dataset="site", load_model=load_model, plans=False)


## Part 1 · Meet the photos

Detection (MP3) draws a **box** around an object. **Segmentation** goes one step further: it decides, *pixel by pixel*, what belongs to the object. That is what makes it useful for measurement: count the pixels of a material and you have its share of the photo; know the scale and you have an area.

The model in this notebook is **SAM 3** (Segment Anything Model 3, Meta 2025). You do not train it. You type a short phrase, such as *concrete* or *steel reinforcement bars*, and it returns every region in the photo that matches, each with a **confidence**.

Photos of foundations, frames and concrete pours, from a 1937 public-domain project record to sites of the 2020s. All from Wikimedia Commons under open licences (credits at the bottom of the notebook).

In [ ]:
#@title ▶ Step 1a · Browse the photos { display-mode: "form" }
#@markdown *all* shows every photo with its credit. A series letter shows one site through time.
which = "all" #@param ["all", "A", "B"]
lab.show_photos(which)


## Part 2 · Segment a material by name

Pick a photo and a material. You get three panels: the photo, the **mask** (white = the model says *this is it*), and the **overlay**. The share of the photo covered by the mask is printed below, together with the confidence of each region. The **confidence slider** hides the regions the model is unsure about: watch how the share changes.

In [ ]:
#@title ▶ Step 2a · Original → mask → overlay { display-mode: "form" }
#@markdown Try several materials on the same photo, then the same material on other photos.
photo = "site_01: Formwork and rebar before a concrete pour, East Approach, New York, 2017" #@param ["site_01: Formwork and rebar before a concrete pour, East Approach, New York, 2017", "site_02: Setting a rebar cage for a foundation, Queens, 2018", "site_03: Rebar on a building site, Hölzla", "site_04: Rebar on a building site, Hölzla (2)", "site_05: Rebar for bridge foundation piling", "site_06: Builders tie a rebar cage together", "site_07: Pouring concrete footings, Trimingham, 8 Jan 2021", "site_08: Brick and block footings, Trimingham, 13 Jan 2021", "site_09: Foundations and concrete oversite, Trimingham, 18 Jan 2021", "site_10: Foundations and concrete oversite, Trimingham, 18 Jan 2021 (3)", "site_11: Concrete pouring for a new school, Spangdahlem", "site_12: Concrete pouring for a new school, Spangdahlem (2)", "site_13: Construction site with a concrete pump truck", "site_14: Seabees pour concrete at a project site, Senegal", "site_15: Workers pouring concrete from a boom pump", "site_16: Construction of a new supermarket, Biddulph, 2010", "site_18: Constructions and scaffolding at the Oosterdokseiland, Amsterdam", "site_19: A building frame with scaffolding at a construction site", "site_20: The Milliners apartment building under construction, September 2025", "site_21: High-rise construction with scaffolding and protective netting, Makati", "site_22: Hay Street parking deck under construction", "site_23: Belchertown administration building: foundation, 30 Aug 1937", "site_24: Belchertown: forms and reinforcing for the foundation, 10 Sep 1937", "site_25: Belchertown: foundation, 11 Oct 1937", "site_26: Belchertown: general view, 2 Nov 1937", "site_27: Belchertown: steel for the main building, 30 Nov 1937", "site_28: Belchertown: laying brick, 20 Dec 1937", "site_29: Belchertown: looking south-east at the main building, 27 Jan 1938", "site_30: Belchertown: main building, 12 May 1938", "site_31: Belchertown: administration buildings, 10 Aug 1938", "site_32: Belchertown: finished administration building, 14 Sep 1939"]
material = "concrete" #@param ["concrete", "rebar (reinforcing steel)", "formwork", "scaffolding", "structural steel", "brick / blockwork", "soil / ground", "timber", "worker", "machine / vehicle", "sky"]
threshold = 0.5 #@param {type:"slider", min:0.2, max:0.9, step:0.05}
lab.segment(photo, material, threshold)


In [ ]:
#@title ▶ Step 2b · The material mix of one photo { display-mode: "form" }
#@markdown Every material at once, with the share of the photo each one covers. Regions may overlap (a worker standing in front of concrete), so the shares need not add up to 100.
photo = "site_01: Formwork and rebar before a concrete pour, East Approach, New York, 2017" #@param ["site_01: Formwork and rebar before a concrete pour, East Approach, New York, 2017", "site_02: Setting a rebar cage for a foundation, Queens, 2018", "site_03: Rebar on a building site, Hölzla", "site_04: Rebar on a building site, Hölzla (2)", "site_05: Rebar for bridge foundation piling", "site_06: Builders tie a rebar cage together", "site_07: Pouring concrete footings, Trimingham, 8 Jan 2021", "site_08: Brick and block footings, Trimingham, 13 Jan 2021", "site_09: Foundations and concrete oversite, Trimingham, 18 Jan 2021", "site_10: Foundations and concrete oversite, Trimingham, 18 Jan 2021 (3)", "site_11: Concrete pouring for a new school, Spangdahlem", "site_12: Concrete pouring for a new school, Spangdahlem (2)", "site_13: Construction site with a concrete pump truck", "site_14: Seabees pour concrete at a project site, Senegal", "site_15: Workers pouring concrete from a boom pump", "site_16: Construction of a new supermarket, Biddulph, 2010", "site_18: Constructions and scaffolding at the Oosterdokseiland, Amsterdam", "site_19: A building frame with scaffolding at a construction site", "site_20: The Milliners apartment building under construction, September 2025", "site_21: High-rise construction with scaffolding and protective netting, Makati", "site_22: Hay Street parking deck under construction", "site_23: Belchertown administration building: foundation, 30 Aug 1937", "site_24: Belchertown: forms and reinforcing for the foundation, 10 Sep 1937", "site_25: Belchertown: foundation, 11 Oct 1937", "site_26: Belchertown: general view, 2 Nov 1937", "site_27: Belchertown: steel for the main building, 30 Nov 1937", "site_28: Belchertown: laying brick, 20 Dec 1937", "site_29: Belchertown: looking south-east at the main building, 27 Jan 1938", "site_30: Belchertown: main building, 12 May 1938", "site_31: Belchertown: administration buildings, 10 Aug 1938", "site_32: Belchertown: finished administration building, 14 Sep 1939"]
threshold = 0.5 #@param {type:"slider", min:0.2, max:0.9, step:0.05}
lab.material_mix(photo, threshold)


In [ ]:
#@title ▶ Step 2c · Can *you* estimate the share? { display-mode: "form" }
#@markdown A photo and a material: guess how much of the photo it covers, then see what SAM 3 measures.
rounds = 4 #@param {type:"slider", min:2, max:8, step:1}
lab.guess_game(rounds)


> ### 📝 Report question 1
> What was your score in the estimation game? Pick one photo and give its material mix at confidence 0.5 (the numbers from Step 2b). Then move the threshold to 0.3 and 0.8 for one material: how much does the share change, and why?

## Part 3 · Compare photos and follow progress

Two photos of the same site at different times tell a story: formwork and rebar disappear, concrete and brick appear. Comparing the material shares turns that story into numbers. Keep in mind what the number is: the share of the **photo**, not the share of the **work**: camera position, zoom and the sky change it without any progress on site.

In [ ]:
#@title ▶ Step 3a · Compare two photos { display-mode: "form" }
photo_a = "site_07: Pouring concrete footings, Trimingham, 8 Jan 2021" #@param ["site_01: Formwork and rebar before a concrete pour, East Approach, New York, 2017", "site_02: Setting a rebar cage for a foundation, Queens, 2018", "site_03: Rebar on a building site, Hölzla", "site_04: Rebar on a building site, Hölzla (2)", "site_05: Rebar for bridge foundation piling", "site_06: Builders tie a rebar cage together", "site_07: Pouring concrete footings, Trimingham, 8 Jan 2021", "site_08: Brick and block footings, Trimingham, 13 Jan 2021", "site_09: Foundations and concrete oversite, Trimingham, 18 Jan 2021", "site_10: Foundations and concrete oversite, Trimingham, 18 Jan 2021 (3)", "site_11: Concrete pouring for a new school, Spangdahlem", "site_12: Concrete pouring for a new school, Spangdahlem (2)", "site_13: Construction site with a concrete pump truck", "site_14: Seabees pour concrete at a project site, Senegal", "site_15: Workers pouring concrete from a boom pump", "site_16: Construction of a new supermarket, Biddulph, 2010", "site_18: Constructions and scaffolding at the Oosterdokseiland, Amsterdam", "site_19: A building frame with scaffolding at a construction site", "site_20: The Milliners apartment building under construction, September 2025", "site_21: High-rise construction with scaffolding and protective netting, Makati", "site_22: Hay Street parking deck under construction", "site_23: Belchertown administration building: foundation, 30 Aug 1937", "site_24: Belchertown: forms and reinforcing for the foundation, 10 Sep 1937", "site_25: Belchertown: foundation, 11 Oct 1937", "site_26: Belchertown: general view, 2 Nov 1937", "site_27: Belchertown: steel for the main building, 30 Nov 1937", "site_28: Belchertown: laying brick, 20 Dec 1937", "site_29: Belchertown: looking south-east at the main building, 27 Jan 1938", "site_30: Belchertown: main building, 12 May 1938", "site_31: Belchertown: administration buildings, 10 Aug 1938", "site_32: Belchertown: finished administration building, 14 Sep 1939"]
photo_b = "site_09: Foundations and concrete oversite, Trimingham, 18 Jan 2021" #@param ["site_01: Formwork and rebar before a concrete pour, East Approach, New York, 2017", "site_02: Setting a rebar cage for a foundation, Queens, 2018", "site_03: Rebar on a building site, Hölzla", "site_04: Rebar on a building site, Hölzla (2)", "site_05: Rebar for bridge foundation piling", "site_06: Builders tie a rebar cage together", "site_07: Pouring concrete footings, Trimingham, 8 Jan 2021", "site_08: Brick and block footings, Trimingham, 13 Jan 2021", "site_09: Foundations and concrete oversite, Trimingham, 18 Jan 2021", "site_10: Foundations and concrete oversite, Trimingham, 18 Jan 2021 (3)", "site_11: Concrete pouring for a new school, Spangdahlem", "site_12: Concrete pouring for a new school, Spangdahlem (2)", "site_13: Construction site with a concrete pump truck", "site_14: Seabees pour concrete at a project site, Senegal", "site_15: Workers pouring concrete from a boom pump", "site_16: Construction of a new supermarket, Biddulph, 2010", "site_18: Constructions and scaffolding at the Oosterdokseiland, Amsterdam", "site_19: A building frame with scaffolding at a construction site", "site_20: The Milliners apartment building under construction, September 2025", "site_21: High-rise construction with scaffolding and protective netting, Makati", "site_22: Hay Street parking deck under construction", "site_23: Belchertown administration building: foundation, 30 Aug 1937", "site_24: Belchertown: forms and reinforcing for the foundation, 10 Sep 1937", "site_25: Belchertown: foundation, 11 Oct 1937", "site_26: Belchertown: general view, 2 Nov 1937", "site_27: Belchertown: steel for the main building, 30 Nov 1937", "site_28: Belchertown: laying brick, 20 Dec 1937", "site_29: Belchertown: looking south-east at the main building, 27 Jan 1938", "site_30: Belchertown: main building, 12 May 1938", "site_31: Belchertown: administration buildings, 10 Aug 1938", "site_32: Belchertown: finished administration building, 14 Sep 1939"]
threshold = 0.5 #@param {type:"slider", min:0.2, max:0.9, step:0.05}
lab.compare(photo_a, photo_b, threshold)


In [ ]:
#@title ▶ Step 3b · One site through time { display-mode: "form" }
#@markdown The photos of a series in order, and a chart of each material's share over time.
series = "A" #@param ["A", "B"]
threshold = 0.5 #@param {type:"slider", min:0.2, max:0.9, step:0.05}
lab.series(series, threshold)


> ### 📝 Report question 2
> From Step 3b: which materials rise and which fall over the series, and does that match what a site manager would expect? Give one example where the number changes for a reason that has nothing to do with progress (camera position, sky, an old black-and-white photo...).

## Part 4 · Where does it go wrong?

Three kinds of error to look for: the **words** you use (the model was trained on everyday language, not construction jargon), **weak regions** the model proposes with low confidence, and plain **mistakes** that need a correction. The last step lets you correct the model by drawing a box over what it got wrong.

In [ ]:
#@title ▶ Step 4a · Does the wording matter? { display-mode: "form" }
#@markdown The same material asked for with different words. Alternative wordings are precomputed for site_01, site_07, site_11, site_14, site_18, site_27; other photos need the live model.
photo = "site_01: Formwork and rebar before a concrete pour, East Approach, New York, 2017" #@param ["site_01: Formwork and rebar before a concrete pour, East Approach, New York, 2017", "site_02: Setting a rebar cage for a foundation, Queens, 2018", "site_03: Rebar on a building site, Hölzla", "site_04: Rebar on a building site, Hölzla (2)", "site_05: Rebar for bridge foundation piling", "site_06: Builders tie a rebar cage together", "site_07: Pouring concrete footings, Trimingham, 8 Jan 2021", "site_08: Brick and block footings, Trimingham, 13 Jan 2021", "site_09: Foundations and concrete oversite, Trimingham, 18 Jan 2021", "site_10: Foundations and concrete oversite, Trimingham, 18 Jan 2021 (3)", "site_11: Concrete pouring for a new school, Spangdahlem", "site_12: Concrete pouring for a new school, Spangdahlem (2)", "site_13: Construction site with a concrete pump truck", "site_14: Seabees pour concrete at a project site, Senegal", "site_15: Workers pouring concrete from a boom pump", "site_16: Construction of a new supermarket, Biddulph, 2010", "site_18: Constructions and scaffolding at the Oosterdokseiland, Amsterdam", "site_19: A building frame with scaffolding at a construction site", "site_20: The Milliners apartment building under construction, September 2025", "site_21: High-rise construction with scaffolding and protective netting, Makati", "site_22: Hay Street parking deck under construction", "site_23: Belchertown administration building: foundation, 30 Aug 1937", "site_24: Belchertown: forms and reinforcing for the foundation, 10 Sep 1937", "site_25: Belchertown: foundation, 11 Oct 1937", "site_26: Belchertown: general view, 2 Nov 1937", "site_27: Belchertown: steel for the main building, 30 Nov 1937", "site_28: Belchertown: laying brick, 20 Dec 1937", "site_29: Belchertown: looking south-east at the main building, 27 Jan 1938", "site_30: Belchertown: main building, 12 May 1938", "site_31: Belchertown: administration buildings, 10 Aug 1938", "site_32: Belchertown: finished administration building, 14 Sep 1939"]
material = "rebar (reinforcing steel)" #@param ["concrete", "rebar (reinforcing steel)", "formwork", "scaffolding", "structural steel", "brick / blockwork", "soil / ground", "timber", "worker", "machine / vehicle", "sky"]
threshold = 0.5 #@param {type:"slider", min:0.2, max:0.9, step:0.05}
lab.phrase_lab(photo, material, threshold)


In [ ]:
#@title ▶ Step 4b · Look at each region and its confidence { display-mode: "form" }
#@markdown Every region the model proposed, numbered, with its confidence. Move the slider to see which ones survive.
photo = "site_01: Formwork and rebar before a concrete pour, East Approach, New York, 2017" #@param ["site_01: Formwork and rebar before a concrete pour, East Approach, New York, 2017", "site_02: Setting a rebar cage for a foundation, Queens, 2018", "site_03: Rebar on a building site, Hölzla", "site_04: Rebar on a building site, Hölzla (2)", "site_05: Rebar for bridge foundation piling", "site_06: Builders tie a rebar cage together", "site_07: Pouring concrete footings, Trimingham, 8 Jan 2021", "site_08: Brick and block footings, Trimingham, 13 Jan 2021", "site_09: Foundations and concrete oversite, Trimingham, 18 Jan 2021", "site_10: Foundations and concrete oversite, Trimingham, 18 Jan 2021 (3)", "site_11: Concrete pouring for a new school, Spangdahlem", "site_12: Concrete pouring for a new school, Spangdahlem (2)", "site_13: Construction site with a concrete pump truck", "site_14: Seabees pour concrete at a project site, Senegal", "site_15: Workers pouring concrete from a boom pump", "site_16: Construction of a new supermarket, Biddulph, 2010", "site_18: Constructions and scaffolding at the Oosterdokseiland, Amsterdam", "site_19: A building frame with scaffolding at a construction site", "site_20: The Milliners apartment building under construction, September 2025", "site_21: High-rise construction with scaffolding and protective netting, Makati", "site_22: Hay Street parking deck under construction", "site_23: Belchertown administration building: foundation, 30 Aug 1937", "site_24: Belchertown: forms and reinforcing for the foundation, 10 Sep 1937", "site_25: Belchertown: foundation, 11 Oct 1937", "site_26: Belchertown: general view, 2 Nov 1937", "site_27: Belchertown: steel for the main building, 30 Nov 1937", "site_28: Belchertown: laying brick, 20 Dec 1937", "site_29: Belchertown: looking south-east at the main building, 27 Jan 1938", "site_30: Belchertown: main building, 12 May 1938", "site_31: Belchertown: administration buildings, 10 Aug 1938", "site_32: Belchertown: finished administration building, 14 Sep 1939"]
material = "concrete" #@param ["concrete", "rebar (reinforcing steel)", "formwork", "scaffolding", "structural steel", "brick / blockwork", "soil / ground", "timber", "worker", "machine / vehicle", "sky"]
lab.inspect(photo, material)


In [ ]:
#@title ▶ Step 4c · Correct it with a box { display-mode: "form" }
#@markdown Draw a box over a region that is wrong, click *Submit*: SAM 3 runs again with your box as a *not this* hint. Needs the live model.
photo = "site_01: Formwork and rebar before a concrete pour, East Approach, New York, 2017" #@param ["site_01: Formwork and rebar before a concrete pour, East Approach, New York, 2017", "site_02: Setting a rebar cage for a foundation, Queens, 2018", "site_03: Rebar on a building site, Hölzla", "site_04: Rebar on a building site, Hölzla (2)", "site_05: Rebar for bridge foundation piling", "site_06: Builders tie a rebar cage together", "site_07: Pouring concrete footings, Trimingham, 8 Jan 2021", "site_08: Brick and block footings, Trimingham, 13 Jan 2021", "site_09: Foundations and concrete oversite, Trimingham, 18 Jan 2021", "site_10: Foundations and concrete oversite, Trimingham, 18 Jan 2021 (3)", "site_11: Concrete pouring for a new school, Spangdahlem", "site_12: Concrete pouring for a new school, Spangdahlem (2)", "site_13: Construction site with a concrete pump truck", "site_14: Seabees pour concrete at a project site, Senegal", "site_15: Workers pouring concrete from a boom pump", "site_16: Construction of a new supermarket, Biddulph, 2010", "site_18: Constructions and scaffolding at the Oosterdokseiland, Amsterdam", "site_19: A building frame with scaffolding at a construction site", "site_20: The Milliners apartment building under construction, September 2025", "site_21: High-rise construction with scaffolding and protective netting, Makati", "site_22: Hay Street parking deck under construction", "site_23: Belchertown administration building: foundation, 30 Aug 1937", "site_24: Belchertown: forms and reinforcing for the foundation, 10 Sep 1937", "site_25: Belchertown: foundation, 11 Oct 1937", "site_26: Belchertown: general view, 2 Nov 1937", "site_27: Belchertown: steel for the main building, 30 Nov 1937", "site_28: Belchertown: laying brick, 20 Dec 1937", "site_29: Belchertown: looking south-east at the main building, 27 Jan 1938", "site_30: Belchertown: main building, 12 May 1938", "site_31: Belchertown: administration buildings, 10 Aug 1938", "site_32: Belchertown: finished administration building, 14 Sep 1939"]
material = "concrete" #@param ["concrete", "rebar (reinforcing steel)", "formwork", "scaffolding", "structural steel", "brick / blockwork", "soil / ground", "timber", "worker", "machine / vehicle", "sky"]
threshold = 0.5 #@param {type:"slider", min:0.2, max:0.9, step:0.05}
lab.fix(photo, material, threshold)


In [ ]:
#@title ▶ Step 4d · Your own words { display-mode: "form" }
#@markdown Type any phrase: a material, a tool, a machine, a colour. Needs the live model.
photo = "site_01: Formwork and rebar before a concrete pour, East Approach, New York, 2017" #@param ["site_01: Formwork and rebar before a concrete pour, East Approach, New York, 2017", "site_02: Setting a rebar cage for a foundation, Queens, 2018", "site_03: Rebar on a building site, Hölzla", "site_04: Rebar on a building site, Hölzla (2)", "site_05: Rebar for bridge foundation piling", "site_06: Builders tie a rebar cage together", "site_07: Pouring concrete footings, Trimingham, 8 Jan 2021", "site_08: Brick and block footings, Trimingham, 13 Jan 2021", "site_09: Foundations and concrete oversite, Trimingham, 18 Jan 2021", "site_10: Foundations and concrete oversite, Trimingham, 18 Jan 2021 (3)", "site_11: Concrete pouring for a new school, Spangdahlem", "site_12: Concrete pouring for a new school, Spangdahlem (2)", "site_13: Construction site with a concrete pump truck", "site_14: Seabees pour concrete at a project site, Senegal", "site_15: Workers pouring concrete from a boom pump", "site_16: Construction of a new supermarket, Biddulph, 2010", "site_18: Constructions and scaffolding at the Oosterdokseiland, Amsterdam", "site_19: A building frame with scaffolding at a construction site", "site_20: The Milliners apartment building under construction, September 2025", "site_21: High-rise construction with scaffolding and protective netting, Makati", "site_22: Hay Street parking deck under construction", "site_23: Belchertown administration building: foundation, 30 Aug 1937", "site_24: Belchertown: forms and reinforcing for the foundation, 10 Sep 1937", "site_25: Belchertown: foundation, 11 Oct 1937", "site_26: Belchertown: general view, 2 Nov 1937", "site_27: Belchertown: steel for the main building, 30 Nov 1937", "site_28: Belchertown: laying brick, 20 Dec 1937", "site_29: Belchertown: looking south-east at the main building, 27 Jan 1938", "site_30: Belchertown: main building, 12 May 1938", "site_31: Belchertown: administration buildings, 10 Aug 1938", "site_32: Belchertown: finished administration building, 14 Sep 1939"]
phrase = "safety helmet" #@param {type:"string"}
threshold = 0.5 #@param {type:"slider", min:0.2, max:0.9, step:0.05}
lab.your_phrase(photo, phrase, threshold)


> ### 📝 Report question 3
> From Step 4a: which wording gave the most sensible mask for the material you chose, and how far apart were the shares? Why would *rebar* and *steel reinforcement bars* give different answers?

> ### 📝 Report question 4
> Describe one mistake you found in Step 4b or 4c (what was included or missed, at which confidence). Did the negative box fix it? What would you tell a colleague who wants to use these percentages in a progress report?

## Part 5 · Your own photo

In [ ]:
#@title ▶ Your photo, your words { display-mode: "form" }
#@markdown Upload a photo (or open the public link on your phone), type what to find, move the threshold.
#@markdown Test at least 1 photo(s) of your own and take screenshots for your report. Needs the live model.
lab.upload_app()


> ### 📝 Report question 5
> Test 1 photo(s) of your own (walls, floors, a site, a street). For each: the phrase you used, the share measured, and whether the mask is right. What kind of surface or wording failed?

> ### 📝 Report question 6
> Where on a project would a measurement like *share of the photo covered by X* be useful, and where would it mislead? What would you need (camera position, reference lengths, drawings, several photos) to turn it into a real quantity?

## Wrap-up

In [ ]:
#@title ▶ Numbers for your report { display-mode: "form" }
lab.report_summary()


### Photo credits and model
All photos are from Wikimedia Commons under the licence shown with each photo in Step 1a (public domain, CC0, CC BY or CC BY-SA; credits are also in `data/photos/*/credits.json`). Plan drawings are course material.

- Model: SAM 3 by Meta AI (SAM License), loaded from a public mirror of the official checkpoint; a copy of the licence is in `docs/SAM_LICENSE.txt`.
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp4_segmentation`).